In [10]:
import os
import time
import cv2
import numpy as np
import vision.utils.box_utils_numpy as box_utils
import onnxruntime as ort

# 定义预测函数，对模型输出的边界框和置信度进行后处理
# confidences: 模型输出的置信度，形状为(1, num_boxes, num_classes)
# boxes: 模型输出的边界框，形状为(1, num_boxes, 4)
def predict(width, height, confidences, boxes, prob_threshold, iou_threshold=0.3, top_k=-1):
    boxes = boxes[0] # boxes形状是 (4420, 4)，表示有4420个预测边界框
    confidences = confidences[0] # confidences形状是 (4420, 2)，表示每个边界框对应2个类别的置信度分数
    picked_box_probs = []
    picked_labels = []
    for class_index in range(1, confidences.shape[1]): # 遍历每个类别，从1开始，跳过了类别0
        probs = confidences[:, class_index] # 获取当前类别的所有置信度分数，得到的是一维数组，形状是(4420,)
        mask = probs > prob_threshold #创建一个布尔掩码，找到置信度大于阈值的边界框
        probs = probs[mask] # 只保留满足阈值的置信度
        if probs.shape[0] == 0: 
            continue
        subset_boxes = boxes[mask, :]  # 获取满足条件的边界框，形状就是(n, 4)
        # .reshape(-1, 1)将一维数组转换为二维列向量，-1表示"自动计算这个维度的大小"
        # 将边界框坐标和置信度拼接成一个数组，形状为 (N, 5)，其中：前4列是边界框坐标第5列是置信度分数
        box_probs = np.concatenate([subset_boxes, probs.reshape(-1, 1)], axis=1) 
        box_probs = box_utils.hard_nms(box_probs,
                                       iou_threshold=iou_threshold,
                                       top_k=top_k,
                                       ) # 对当前类别的边界框进行非极大值抑制（NMS）： 移除高度重叠的边界框，只保留置信度最高的边界框，top_k=-1表示保留所有满足条件的边界框
        #将NMS后的结果添加到列表中，并记录对应的类别标签
        picked_box_probs.append(box_probs) 
        # [class_index] * box_probs.shape[0]列表乘法操作，会创建一个长度为box_probs.shape[0]的新列表，其中每个元素都是 class_index
        # extend()方法将列表中的每个元素添加到目标列表中。最终结果：
        # 类别标签列表: [1, 1, 1, 3, 3]
        # 对应边界框:   [框1, 框2, 框3, 框4, 框5]
        # 对应关系:      类别1 类别1 类别1 类别3 类别3
        picked_labels.extend([class_index] * box_probs.shape[0])
    if not picked_box_probs:
        return np.array([]), np.array([]), np.array([])
    
    # picked_box_probs = [
    # array([[x1, y1, x2, y2, score],  # 类别1的3个框
    #       [...],
    #       [...]]),                   # 形状: (3, 5)
    #
    # array([[x1, y1, x2, y2, score],  # 类别2的2个框
    #       [...]]),                   # 形状: (2, 5)
    #
    # array([[x1, y1, x2, y2, score],  # 类别3的4个框
    #       [...],
    #       [...],
    #       [...]]),                   # 形状: (4, 5)
    # ]
    # 这是一个列表，包含3个独立的NumPy数组。将所有类别的结果拼接成一个数组
    picked_box_probs = np.concatenate(picked_box_probs)
    # 边界框坐标反归一化，模型输出的是归一化坐标（0-1之间）；需要乘以图像的实际宽度和高度；假设坐标格式是：[x_min, y_min, x_max, y_max]
    picked_box_probs[:, 0] *= width
    picked_box_probs[:, 1] *= height
    picked_box_probs[:, 2] *= width
    picked_box_probs[:, 3] *= height
    # 返回三个结果：边界框坐标（整数，像素坐标），类别标签，置信度分数
    return picked_box_probs[:, :4].astype(np.int32), np.array(picked_labels), picked_box_probs[:, 4]

# 从标签文件中读取每一行，并去除行首尾的空白字符，得到类别名称列表 2分
class_names = [name.strip() for name in open('voc-model-labels.txt').readlines()]

# 创建 ONNX Runtime 的推理会话，用于运行模型进行推理 2分
ort_session = ort.InferenceSession('version-RFB-320.onnx')

# 获取模型输入的名称 2分
input_name = ort_session.get_inputs()[0].name

# 定义保存检测结果图像的目录路径
result_path = "./detect_imgs_results_onnx"

# 定义置信度阈值，用于筛选出置信度较高的检测结果
threshold = 0.7
# 定义存储待检测图像的目录路径
path = "imgs"
# 用于统计所有图像中检测到的目标框总数，初始化为 0
sum = 0

# 如果保存结果的目录不存在，则创建该目录 2分
if not os.path.exists(result_path):
    os.makedirs(result_path)
    
# 获取指定目录下的所有文件和文件夹名称列表
listdir = os.listdir(path)

# 遍历目录下的每个文件
for file_path in listdir:
    # 拼接图像文件的完整路径
    img_path = os.path.join(path, file_path)
    # 使用 OpenCV 读取图像文件 2分
    orig_image = cv2.imread(img_path)
    # 将图像从 BGR 颜色空间转换为 RGB 颜色空间（许多模型要求输入为 RGB 格式）
    image = cv2.cvtColor(orig_image, cv2.COLOR_BGR2RGB)
    # 将图像调整为 320x240 的尺寸（符合模型输入的尺寸要求） 2分
    image = cv2.resize(image, (320, 240))
    # 定义图像归一化的均值数组 2分
    image_mean = np.array([127, 127, 127])
    # 对图像进行归一化处理，减去均值并除以 128
    image = (image - image_mean) / 128
    # 将图像的维度从 (高度, 宽度, 通道数) 转换为 (通道数, 高度, 宽度)
    image = np.transpose(image, [2, 0, 1])
    # 在第一个维度上扩展一个维度，将图像变为 (1, 通道数, 高度, 宽度)，以符合模型输入的维度要求  1分
    image = np.expand_dims(image, axis=0)
    # 将图像数据类型转换为 float32 类型
    image = image.astype(np.float32)
    # 记录开始时间，用于计算模型推理的耗时
    time_time = time.time()
    # 使用 ONNX Runtime 运行模型，输入图像数据，得到模型输出的置信度和边界框  2分
    confidences, boxes = ort_session.run(None, {input_name: image})
    
    print(boxes.shape)
    print(confidences.shape)
    
    # 计算并打印模型推理的耗时
    print("cost time:{}".format(time.time() - time_time))
    # 调用 predict 函数对模型输出的边界框和置信度进行后处理，得到最终的边界框、类别标签和置信度
    boxes, labels, probs = predict(orig_image.shape[1], orig_image.shape[0], confidences, boxes, threshold)
    # 遍历每个检测到的目标框
    for i in range(boxes.shape[0]):
        # 获取当前目标框的坐标
        box = boxes[i, :]
        # 生成当前目标框的标签字符串，包含类别名称和置信度
        label = f"{class_names[labels[i]]}: {probs[i]:.2f}"

        # 在原始图像上绘制目标框，颜色为 (255, 255, 0)，线条粗细为 4
        # (box[0], box[1])​ - 矩形左上角坐标
        # (box[2], box[3])​ - 矩形右下角坐标
        # (255, 255, 0)​ - 矩形颜色 OpenCV使用BGR颜色顺序
        # 4​ - 线宽/线粗
        cv2.rectangle(orig_image, (box[0], box[1]), (box[2], box[3]), (255, 255, 0), 4)
        # 将绘制了目标框的图像保存到结果目录中
        cv2.imwrite(os.path.join(result_path, file_path), orig_image)
    # 累加当前图像中检测到的目标框数量到总数中
    sum += boxes.shape[0]
# 打印所有图像中检测到的目标框总数
print("sum:{}".format(sum))

(1, 4420, 4)
(1, 4420, 2)
cost time:0.00941610336303711
sum:4


In [7]:
help(os.makedirs)

Help on function makedirs in module os:

makedirs(name, mode=511, exist_ok=False)
    makedirs(name [, mode=0o777][, exist_ok=False])
    
    Super-mkdir; create a leaf directory and all intermediate ones.  Works like
    mkdir, except that any intermediate path segment (not just the rightmost)
    will be created if it does not exist. If the target directory already
    exists, raise an OSError if exist_ok is False. Otherwise no exception is
    raised.  This is recursive.



In [8]:
print(boxes.shape)
print(confidences.shape)

(4, 4)
(1, 4420, 2)
